# Session 3 — Downstream evaluation and attention maps

Runs the real FWI evaluation: each pretrained network is used as the
reparameterization ansatz and driven through the full inversion loop.
**Requires session 2's checkpoints.**

## 1. Setup

In [ ]:
!git clone -b improve_transformer https://github.com/rifatozkurt/FullWaveformInversion
%cd FullWaveformInversion

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'{p.name}, {p.total_memory/1e9:.1f} GB')
    # A single adjoint evaluation allocates ~3 GB. Anything under ~8 GB
    # means running one job at a time and nothing else on the GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/fwi_thesis'
!mkdir -p {OUT}

## 2. Data

`extended/` (ids 0-14999) is training data; `eval/` (ids 15000-15999) is
held out. They do NOT overlap. Restore both from Drive if you have them
zipped there, otherwise generate (slow).

In [ ]:
# Restore from Drive (fast path)
!unzip -q -o {OUT}/data/extended.zip -d /content/  || echo 'no extended.zip'
!unzip -q -o {OUT}/data/eval.zip     -d /content/  || echo 'no eval.zip'
!ls /content/extended | head -3 ; ls /content/eval | head -3

# --- alternative: generate instead (hours) ---
# !python scripts/generate_train_data_colab.py \
#     --config configs/config_final.yaml --output-dir /content/extended \
#     --start-case-id 0 --number-of-cases 15000 --case-batch-size 4 --no-overwrite

### Restore session 2's checkpoints

This session cannot run without them.

In [ ]:
!mkdir -p models
!cp -r {OUT}/session2/models/final models/ 2>/dev/null || unzip -q -o {OUT}/session2/models.zip -d .
!ls models/final | head -20
import pathlib
n = len(list(pathlib.Path('models/final').glob('*.pt'))) + \
    len(list(pathlib.Path('models/final').glob('model_Unet*')))
print(f'{n} checkpoints found')
assert n > 0, 'No checkpoints restored -- session 3 cannot run.'

## 3a. U-Net vs SegFormer, downstream FWI

Every checkpoint inverted on 6 held-out cases. Note the eval cases have
very different void fractions (0.18%–4.14%), so per-case numbers are
written to CSV, not just the mean.

In [ ]:
!python scripts/compare_unet_transformer.py \
    --config configs/config_final.yaml \
    --data-dir /content/eval \
    --model-dir models/final \
    --cases 15000,15001,15002,15003,15004,15005 \
    --sample-counts 250,500,1000,5000,10000,15000 \
    --models unet,segformer \
    --run-dir runs/final/compare_unet_segformer

## 3b. ImageNet vs random initialization

Same architecture, same data, same recipe — only the initial encoder
weights differ.

In [ ]:
!python scripts/compare_unet_transformer.py \
    --config configs/config_final.yaml \
    --data-dir /content/eval \
    --model-dir models/final \
    --cases 15000,15001,15002,15003,15004,15005 \
    --sample-counts 15000 \
    --models segformer,segformer_imagenet \
    --run-dir runs/final/compare_imagenet

## 3c. Attention maps

Caveat for the write-up: SegFormer reduces keys by the SR ratio, so each
block attends to only ~32 key locations. These are NOT dense per-pixel
ViT maps. What they show is where each output looks and how broadly.

In [ ]:
!python scripts/visualize_segformer_attention.py \
    --config configs/config_final.yaml \
    --data-dir /content/eval \
    --checkpoint models/final/model_SegFormer_100_segmentation_15000.pt \
    --compare-checkpoint models/final/model_SegFormerImageNet_100_segmentation_15000.pt \
    --labels 'SegFormer (random init),SegFormer (ImageNet init)' \
    --case 15000 \
    --output-dir runs/final/attention

## 4. Save everything to Drive

`runs/` holds every history, CSV and figure; `models/` holds the
checkpoints. Zip both so a disconnect does not lose the session.

In [ ]:
SESSION = 'session3'
!mkdir -p {OUT}/{SESSION}
!zip -qr /content/runs.zip runs
!cp /content/runs.zip {OUT}/{SESSION}/runs.zip
!zip -qr /content/models.zip models
!cp /content/models.zip {OUT}/{SESSION}/models.zip
print('saved to', OUT + '/' + SESSION)

## 5. Look at the figures before you disconnect

In [ ]:
from IPython.display import Image, display
import pathlib
for p in sorted(pathlib.Path('runs/final').rglob('report/*.png')):
    print(p)
    display(Image(str(p)))